In [ ]:
import pandas as pd
import numpy as np
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import importlib
import joblib
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
sampling_rate = 100
n_splits = 3
pipe_name = 'imu_extractor'
model_run_name = 'binary_rf.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 
                  'SEQ_000046', 'SEQ_000053', 'SEQ_000058', 'SEQ_000063', 'SEQ_000079']

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)

In [ ]:
# BLOCK 4: Prepare Data with Binary Target
train_df = raw_train_df.set_index('row_id')

# Create binary target (1 = Target, 0 = Non-Target)
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(int)

# Sample data (adjust as needed)
train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.2, test_pct=0.2)

# train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]

print(f"Binary target distribution:")
print(train_sample_df.groupby('sequence_id')['is_target'].first().value_counts())


In [ ]:
importlib.reload(utils)

num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_sample_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
         utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

# BLOCK 6: Feature Extractor
custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

# BLOCK 7: Binary Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle any imbalance
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=rf_clf,
        extractor=custom_extractor,
        mode=None,
        target='is_target'
    ))
])

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain':             ['acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [['rot_w', 'rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes':       [True],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],
    f'{pipe_name}__tof_sensor_list':        [tof_columns], 

    f'{pipe_name}__dc_offset':              [1.5],
    f'{pipe_name}__band_edges':             [log_edges],
    f'{pipe_name}__category_data':          [True],
    f'{pipe_name}__segmentation':           ['window'],

    'classifier__estimator__n_estimators':    [200],
    'classifier__estimator__max_depth':       [5],
    'classifier__estimator__min_samples_split': [5],
}

# BLOCK 9: Grid Search
y = sliced_data_df[['sequence_id', 'is_target']]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring='accuracy',
    verbose=2,
    n_jobs=1,
    return_train_score=True
)

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'binary_rf_results.csv', index=False)

print(f"Best score: {grid_search.best_score_:.4f}")

In [ ]:
# BLOCK 11: Evaluate on Holdout
best_model = grid_search.best_estimator_

extractor = best_model.named_steps['imu_extractor']
preprocessor = best_model.named_steps['preprocessor']
classifier = best_model.named_steps['classifier']

X_feat = extractor.transform(test_sample_df)
X_proc = preprocessor.transform(X_feat)

y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['is_target']
y_true = y_true.reindex(X_proc.index)

y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print(f"ROC AUC: {roc_auc_score(y_true, y_pred):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Binary Classifier')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Feature Importance for Random Forest
best_model = grid_search.best_estimator_

# Get the classifier from pipeline
classifier = best_model.named_steps['classifier'].estimator_

# Get feature names from preprocessor
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# Get importances
importances = classifier.feature_importances_

# Create DataFrame
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feat_imp_df['percentage'] = 100 * (feat_imp_df['importance'] / feat_imp_df['importance'].sum())
feat_imp_df['name'] = feat_imp_df['feature'].str.split('_').str[0]
feat_imp_df = feat_imp_df.sort_values(by='importance', ascending=False).reset_index(drop=True)
core_features_df = feat_imp_df.groupby('name').agg(**{
    'Proportion': ('percentage', 'sum')
}).sort_values(by='Proportion', ascending=False).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), nrows=2, ncols=1)

sns.barplot(data=core_features_df, x='Proportion', y='name', ax=ax[0], orient='h')
ax[0].set_title('Feature Importance')
ax[0].set_xlabel('Proportion (%)')
ax[0].set_ylabel('Feature Group')

sns.barplot(data=feat_imp_df.head(20), x='percentage', y='feature', ax=ax[1], orient='h')
ax[1].set_title('Feature Importance top 20')
ax[1].set_xlabel('Proportion (%)')
ax[1].set_ylabel('Feature')

In [1]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/cnn-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)

2026-04-11 06:31:15.058804: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775889075.298859      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775889075.364591      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775889075.879315      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775889075.879356      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775889075.879358      24 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'cnn_1d_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False

model_target = 'gesture_action'

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.4,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 1944 seqs | 38.0%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

cnn_clf = utils.Keras1DCNNClassifier(verbose=0)
custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('pca', utils.IndexPreservingPCA()),
    ('classifier', utils.ManyToOneWrapper(
        estimator=cnn_clf,
        extractor=custom_extractor,
        mode=None,
        target=model_target
    ))
])

cv = GroupKFold(n_splits=n_splits)

In [6]:
if random_search:
    param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['displacement'],
    f'{pipe_name}__combine_imu_axes':       [False],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [False],
    f'{pipe_name}__rotation_domain':        ['frequency'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['baseline'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['baseline'],

    f'{pipe_name}__window':                 [0.2, 0.5, 1, 2],
    f'{pipe_name}__step_sec':               [0.1, 1],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [linear_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None],

    'classifier__estimator__filters': [48],      
    'classifier__estimator__kernel_size': [7, 9, 11],            
    'classifier__estimator__dropout': [0.14, 0.3],
    'classifier__estimator__learning_rate': [0.001, 0.0001],
    'classifier__estimator__epochs': [50],
    'classifier__estimator__batch_size': [16,],
    }
else:
    param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['displacement'],
    f'{pipe_name}__combine_imu_axes':       [False],
    f'{pipe_name}__sampling_rate':          [100],


    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [False],
    f'{pipe_name}__rotation_domain':        ['frequency'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['baseline'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['baseline'],

    f'{pipe_name}__window':                 [0.2],
    f'{pipe_name}__step_sec':               [0.1],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [linear_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None],

    'classifier__estimator__filters': [48],      
    'classifier__estimator__kernel_size': [7],         
    'classifier__estimator__dropout': [0.3],  
    'classifier__estimator__learning_rate': [0.001],
    'classifier__estimator__epochs': [100],
    'classifier__estimator__batch_size': [16],
}

In [7]:
cv_results_list = []

for col in orientation_cols_dict:
    if random_search:
        search_obj = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            n_iter=5,
            cv=cv,
            random_state=42,
            n_jobs=1,
            verbose=2,
            return_train_score=True
        )
    else:
        search_obj = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=cv,
            verbose=2,
            n_jobs=1,
            return_train_score=True
        )

    sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
    y = sliced_data_df[['sequence_id', model_target]]

    n_fits = n_splits * len(ParameterGrid(param_grid))

    with utils.tqdm_joblib(tqdm(total=n_fits, desc=f"CV {col}")):
        search_obj.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

    if save_model:
        model = search_obj.best_estimator_
        path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
        joblib.dump(model, path_to_model_run_name)

    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['orientation_data'] = col
    cv_results_list.append(cv_results_df)

master_cv_results_df = pd.concat(cv_results_list)
master_cv_results_df['model_target'] = model_target
path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
master_cv_results_df.to_csv(path_to_cv_results, index=False)

CV Lie on Back:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting 3 folds for each of 1 candidates, totalling 3 fits


I0000 00:00:1775889151.571452      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775889151.573982      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775889154.963589      73 service.cc:152] XLA service 0x7ae32400fa90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775889154.963620      73 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775889154.963624      73 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775889155.445358      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775889158.265880      73 device_compiler.h:188] Compiled clust

[CV] END classifier__estimator__batch_size=16, classifier__estimator__dropout=0.3, classifier__estimator__epochs=100, classifier__estimator__filters=48, classifier__estimator__kernel_size=7, classifier__estimator__learning_rate=0.001, imu_extractor__band_edges=[ 0 10 20 30 40 50], imu_extractor__category_data=False, imu_extractor__combine_imu_axes=False, imu_extractor__combine_rot_axes=False, imu_extractor__dc_offset=0, imu_extractor__imu_domain=displacement, imu_extractor__imu_sensor_list=['acc_x', 'acc_y', 'acc_z'], imu_extractor__rotation_domain=frequency, imu_extractor__rotation_sensor_list=['rot_w', 'rot_x', 'rot_y', 'rot_z'], imu_extractor__sampling_rate=100, imu_extractor__segmentation=window, imu_extractor__step_sec=0.1, imu_extractor__thermopile_mode=baseline, imu_extractor__thermopile_sensor_list=['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'], imu_extractor__tof_mode=baseline, imu_extractor__tof_sensor_list=['tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5

In [8]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'cnn_1d_v1_per_gesture_scores.csv')

In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder
from sklearn.pipeline import Pipeline

%load_ext autoreload
%autoreload 2

Basic Model - 30% accuracy achieved

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'

In [ ]:
train_df = raw_train_df.set_index('row_id')
train_df.head(1)

In [ ]:
# Your variables
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
subjects_list = train_df['subject'].unique()

# 1. Create the folder named after your variable
output_dir = f'temp_calculations/{exp_name}'
os.makedirs(output_dir, exist_ok=True)

subject_set_list = []

for single_subject in tqdm(subjects_list, desc='Subject', position=0, leave=True):
    
    # 2. Define the path for this specific subject
    subject_file = f"{output_dir}/{single_subject}.csv"
    
    # 3. CHECK: If file exists, load it and skip the calculation
    if os.path.exists(subject_file):
        temp_processed_subject_df = pd.read_csv(subject_file, index_col=0)
        subject_set_list.append(temp_processed_subject_df)
        continue

    # --- START YOUR ORIGINAL LOGIC ---
    single_subject_df = train_df.loc[train_df['subject'] == single_subject, phase_1_cols]

    sequence_set_list = []
    for single_sequence in tqdm(single_subject_df['sequence_id'].unique(), desc='Sequence', position=1, leave=False):
        single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

        temp_orientation = single_gesture_df['orientation'].unique()[0]
        temp_subject = single_gesture_df['subject'].unique()[0]
        temp_gesture = single_gesture_df['gesture'].unique()[0]

        # Get Acceleration
        temp_acc_time_df = single_gesture_df[['acc_x', 'acc_y', 'acc_z']]
        temp_acc_fft_df = temp_acc_time_df.apply(utils.convert_frame_to_fft, axis=0, args=(sampling_rate,))
        
        # Zeroing DC Offset
        temp_acc_fft_df.loc[0:1] = 0

        features_acc = (
            temp_acc_fft_df
            .apply(lambda col: utils.extract_features(col, sampling_rate=sampling_rate))
            .T
            .add_prefix("")
        )

        single_row_df = features_acc.stack()

        single_row_df.index = [f"{axis}_{feat}" for axis, feat in single_row_df.index]
        single_row_df = single_row_df.to_frame().T
        single_row_df.index = [single_sequence]

        single_row_df.loc[single_sequence, 'orientation'] = temp_orientation
        single_row_df.loc[single_sequence, 'subject'] = temp_subject
        single_row_df.loc[single_sequence, 'gesture'] = temp_gesture

        sequence_set_list.append(single_row_df)

    # Combine the sequences for this subject
    temp_processed_subject_df = pd.concat(sequence_set_list)
    
    # 4. SAVE: Save the subject-level CSV to your experiment folder
    temp_processed_subject_df.to_csv(subject_file)
    
    subject_set_list.append(temp_processed_subject_df)
    # --- END YOUR ORIGINAL LOGIC ---

# Final Merge
feature_df = pd.concat(subject_set_list).merge(train_demo_df, on='subject', how='left')
feature_df.head(1)

In [ ]:
corr_matrix = feature_df[feature_df.columns.difference(['orientation', 'subject', 'gesture'])].corr()

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Vibration Feature Correlation Heatmap')
plt.show()

In [ ]:
categorical_cols = ['orientation']
acc_cols = feature_df.columns[feature_df.columns.str.startswith('acc')].tolist()
demo_cols = train_demo_df.columns[~train_demo_df.columns.str.contains('subject')].tolist()
numerical_cols = acc_cols + demo_cols

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', TargetEncoder(), categorical_cols),
        ('num', 'passthrough', numerical_cols),
        ('drop', 'drop', ['subject'])
    ]
)

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Split data
X = feature_df.drop('gesture', axis=1)
y = feature_df['gesture']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit pipeline
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
X_train

In [ ]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from itertools import combinations
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hist-boost-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils
from scipy.stats import uniform, randint

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV

from scipy.stats import uniform, randint, loguniform

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'hist_boost_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w','rot_x', 'rot_y', 'rot_z']
thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Seated Straight': ['Seated Straight'],
    'Lie on Side': ['Lie on Side - Non Dominant'],
    'Seated Lean': ['Seated Lean Non Dom - FACE DOWN'],
    'Lie on Back': ['Lie on Back'],
    'All': orientation_cols
}

gesture_position_cols_dict = {
    'All': ['Above ear', 'Cheek', 'Eyebrow', 'Eyelash', 'Forehead', 'Neck'],
    'Forehead': ['Forehead'],
    'Neck': ['Neck'],
    'Eyebrow': ['Eyebrow'],
    'Eyelash': ['Eyelash'],
    'Cheek': ['Cheek'],
    'Above ear': ['Above ear'],
     'Forehead and Neck': ['Forehead','Neck'],
}

model_target_list = ['gesture_position', 'gesture_action', 'gesture']

do_report = False
save_model = False
random_search = False

orientation_cols_dict = {
    'All': orientation_cols
}

model_target_list = ['gesture']

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.8,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['velocity'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [True],
    f'{pipe_name}__rotation_domain':        ['orientation'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['spatial'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['blob'],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [log_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None, 0.7], 

    'classifier__estimator__learning_rate': [0.01], 
    'classifier__estimator__max_iter': [100],
    'classifier__estimator__max_depth': [3],
    'classifier__estimator__min_samples_leaf': [14],
    'classifier__estimator__l2_regularization': [1.08],
    'classifier__estimator__max_leaf_nodes': [21],
    'classifier__estimator__early_stopping': [True],
    'classifier__estimator__n_iter_no_change': [10], 
    'classifier__estimator__tol': [1e-4],
}

param_grid[f'{pipe_name}__window'] = [1.2]
param_grid[f'{pipe_name}__step_sec'] = [0.2]

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

# HistGradientBoosting with anti-overfitting defaults
hgb_clf = HistGradientBoostingClassifier(
    loss='log_loss',
    random_state=42,
    verbose=0
)

In [ ]:
for model_target in model_target_list:

    cv_results_list = []
    
    for col in orientation_cols_dict:
        pipeline = Pipeline([
                    (pipe_name, custom_extractor),
                    ('preprocessor', preprocessor),
                    ('pca', utils.IndexPreservingPCA()),
                    ('classifier', utils.ManyToOneWrapper(
                        estimator=hgb_clf,
                        extractor=custom_extractor,
                        mode=None,
                        target=model_target))])

        if random_search:
            search_obj = RandomizedSearchCV(
                            estimator=pipeline,
                            param_distributions=param_grid,
                            n_iter=30,                    # 👈 Only 5 combinations
                            cv=GroupKFold(n_splits=n_splits),                        # 3-fold cross-validation
                            random_state=42,             # Reproducible randomness
                            n_jobs=-1,                   # Use all cores
                            verbose=1,                   # See progress
                            return_train_score=True      # Track overfitting
                        )
        else:
            search_obj = GridSearchCV(
                                estimator=pipeline, 
                                param_grid=param_grid,  
                                cv=GroupKFold(n_splits=n_splits),
                                verbose=1, 
                                n_jobs=-1, 
                                return_train_score=True
                        )
        
        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        search_obj.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
        
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

In [ ]:
if do_report:
    
    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'svm_rbf_per_gesture_scores.csv')

In [ ]:
import pandas as pd
import utils
import os

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']


# Explore

In [ ]:
train_df = raw_train_df.set_index('row_id')
train_df.head(2)

In [ ]:
train_demo_df.head(1)

In [ ]:
train_df['sequence_type'].value_counts()

In [ ]:
train_df['sequence_id'].nunique(), train_df['sequence_counter'].nunique(), train_df['subject'].nunique(), train_df['orientation'].nunique(), train_df['behavior'].nunique()

In [ ]:
train_df['orientation'].value_counts()

In [ ]:
train_df['behavior'].value_counts()

In [ ]:
train_df[feat_columns]

In [ ]:
example_sequence = 'SEQ_000007'
example_seq_df = train_df[train_df['sequence_id'] == example_sequence].merge(train_demo_df, how='left', on='subject')
example_seq_df.groupby(['gesture']).agg(**{
    'Gesture': ('gesture', 'unique'),
    'sequence':('sequence_id', 'first'),
    'subject':('subject', 'unique'),
    'orientation':('orientation', 'unique'),
    'sequence':('sequence_id', 'unique'),
    'sequence_num':('sequence_id', 'count'),
    'behavior_num':('behavior', 'count'),
    'phase':('phase', 'unique'),
    'adult_child':('adult_child', 'first'),
    'age':('age', 'first'),
    'sex':('sex', 'first'),
    'handedness':('handedness', 'first'),
    'height_cm':('height_cm', 'first'),
    'shoulder_to_wrist_cm':('shoulder_to_wrist_cm', 'first'),
    'elbow_to_wrist_cm':('elbow_to_wrist_cm', 'first')
})

In [ ]:
(train_demo_df.groupby(['subject']).agg(**{
    'adult_child':('adult_child','nunique'),
    'age':('age', 'nunique'),
    'sex':('sex', 'nunique'),
    'handedness':('handedness', 'nunique'),
    'height_cm':('height_cm', 'nunique'),
    'shoulder_to_wrist_cm':('shoulder_to_wrist_cm', 'nunique'),
    'elbow_to_wrist_cm':('elbow_to_wrist_cm', 'nunique')
}) > 1).sum().sum()

In [ ]:
example_seq_df.head(1)

In [ ]:
example_seq_df.loc[example_seq_df['sequence_id'] == example_sequence, non_device_cols].head()

In [ ]:
example_seq_df.shape

In [ ]:
train_df.groupby('gesture').agg(**{
    'Recordings': ('sequence_id', 'nunique'),
})

In [ ]:
train_df[train_df['sequence_id'] == 'SEQ_065526'].phase.unique()

In [ ]:
phase_eg = train_df.groupby('sequence_id').agg({'phase': 'nunique','behavior': 'nunique', 'orientation': 'nunique'})
phase_eg[phase_eg['phase'] > 2]

In [ ]:
phase_eg[phase_eg['behavior'] > 3]

In [ ]:
phase_eg[phase_eg['orientation'] > 1]

In [ ]:
train_df[train_df['phase'] == 'Pause']

In [ ]:
train_df['subject'].nunique()

In [ ]:
# Count rows per phase for different sequences
train_df.groupby(['sequence_id', 'behavior']).size()

In [ ]:
example_seq_df[['acc_x', 'acc_y', 'acc_z']].plot()

In [ ]:
example_seq_df[['acc_x', 'acc_y', 'acc_z']].head()

In [ ]:
sampling_rate = 1000
example_fft_df = example_seq_df[['acc_x', 'acc_y', 'acc_z']].apply(utils.convert_frame_to_fft, axis=0, args=(sampling_rate,))
# example_fft_df.loc[0:5] = 0
example_fft_df.head()

In [ ]:
features = (
    example_fft_df
    .apply(lambda col: utils.extract_features(col, sampling_rate=sampling_rate))
    .T
    .add_prefix("")
)

single_row = features.stack()
single_row.index = [f"{axis}_{feat}" for axis, feat in single_row.index]
single_row = single_row.to_frame().T

In [ ]:
import pandas as pd
import numpy as np
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import importlib
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df  = pd.read_csv(data_folder + 'train.csv')
raw_test_df   = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'

model_run_name = 'lgbm.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits   = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013','SEQ_000034', 'SEQ_000046', 'SEQ_000053']

model_target = 'gesture_action'

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target']

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(target_only_df, train_pct=0.2, test_pct=0.2)

# train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_sample_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain':             ['acceleration'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [['rot_w', 'rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes':       [True],
    f'{pipe_name}__rotation_domain':        ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1']],

    f'{pipe_name}__tof_sensor_list':        [tof_1],
    f'{pipe_name}__tof_mode':               ['research'],

    f'{pipe_name}__window':                 [0.2],
    f'{pipe_name}__step_sec':               [0.037],
    f'{pipe_name}__dc_offset':              [1.5],
    f'{pipe_name}__band_edges':             [None],
    f'{pipe_name}__category_data':          [True],
    f'{pipe_name}__segmentation':           ['window'],

    'classifier__estimator__n_estimators':    [300],
    'classifier__estimator__max_depth':       [4],
    'classifier__estimator__learning_rate':   [0.05],
    'classifier__estimator__subsample':       [0.8],
    'classifier__estimator__colsample_bytree':[0.7],
    'classifier__estimator__min_child_samples':[50],
    'classifier__estimator__reg_alpha':       [1],
    'classifier__estimator__reg_lambda':      [10],
    'classifier__estimator__num_leaves':      [8],
    'classifier__estimator__min_gain_to_split':[1.0],  # add this
}

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

lgbm_clf = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=lgbm_clf,
        extractor=custom_extractor,
        mode=None,
        target = model_target
    ))
])

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', model_target]]

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])
pbar.close()

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'lgbm_results.csv', index=False)

In [ ]:
cv_df = cv_results_df.sort_values('rank_test_score').T.reset_index()

split_frame_series = cv_df['index'].str.replace('__', '_').str.split('_')
imu_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'imu')
classifier_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'classifier')

cv_df.loc[split_frame_series.str[-1] == 'time', 'name'] = split_frame_series.str[0:2].str.join('_')
cv_df.loc[split_frame_series.str[-1] == 'time', 'eval'] = 'time'

cv_df.loc[classifier_rows, 'eval'] = 'classifier'
cv_df.loc[classifier_rows, 'name'] = split_frame_series.loc[classifier_rows].str[3:].str.join('_')

cv_df.loc[imu_rows, 'eval'] = 'imu'
cv_df.loc[imu_rows, 'name'] = split_frame_series.loc[imu_rows].str[3:].str.join('_')

cv_df.loc[split_frame_series.str[0] == 'params', 'eval'] = 'params'
cv_df.loc[split_frame_series.str[0] == 'params', 'name'] = 'params'

cv_df.loc[cv_df['index'].str.startswith('split'), 'eval'] = 'split_score'
cv_df.loc[cv_df['index'].str.startswith('split'), 'name'] = split_frame_series.str[0]

cv_df.loc[cv_df['index'] == 'mean_train_score', 'eval'] = 'train_score'
cv_df.loc[cv_df['index'] == 'mean_train_score', 'name'] = 'mean'

cv_df.loc[cv_df['index'] == 'mean_test_score', 'eval'] = 'test_score'
cv_df.loc[cv_df['index'] == 'mean_test_score', 'name'] = 'mean'

cv_df.loc[cv_df['index'] == 'std_train_score', 'eval'] = 'train_score'
cv_df.loc[cv_df['index'] == 'std_train_score', 'name'] = 'std'

cv_df.loc[cv_df['index'] == 'std_test_score', 'eval'] = 'test_score'
cv_df.loc[cv_df['index'] == 'std_test_score', 'name'] = 'std'

cv_df.loc[cv_df['index'] == 'rank_test_score', ['eval', 'name']] = 'rank'
cv_df.drop(columns='index', inplace=True)

to_view = cv_df[cv_df['eval'] != 'classifier']
to_view = to_view[to_view['eval'] != 'time']
to_view = to_view[to_view['name'] != 'std'].set_index(['eval', 'name']).T
to_view

In [ ]:
best_model = grid_search.best_estimator_

extractor    = best_model.named_steps['imu_extractor']
preprocessor = best_model.named_steps['preprocessor']
classifier   = best_model.named_steps['classifier']

X_feat = extractor.transform(test_sample_df)
X_proc = preprocessor.transform(X_feat)

y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
y_true = y_true.reindex(X_proc.index)

y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

print(classification_report(y_true, y_pred))

report_df = pd.DataFrame(
    classification_report(y_true, y_pred, output_dict=True)
).T.sort_values('f1-score', ascending=False)

report_df.to_csv(model_run_folder_name + 'per_gesture_scores.csv')

In [1]:
import os
import warnings
import logging
%pip install -q sktime scikit-optimize sklearn-genetic-opt
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/lgbm-util/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Continuous, Categorical as GenCategorical, Integer as GenInteger
import importlib

from scipy.stats import randint, uniform, loguniform

from lightgbm import LGBMClassifier

from scipy.stats import randint, uniform, loguniform
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as GenCategorical
from sklearn_genetic.space import Integer as GenInteger
from sklearn_genetic.space import Continuous

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

tf.keras.backend.clear_session()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 8.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


E0000 00:00:1777462796.202179      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777462796.278835      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777462796.933556      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933593      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933595      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462796.933597      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100

pipe_name = 'imu_extractor'
model_run_name = 'lgbm_v2.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
tof_1 = [f'tof_1_v{j}' for j in range(64)]

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

model_target = 'gesture_action'
search_mode  = 'grid'  # Options: 'grid', 'random', 'bayesian', 'evolutionary'

candidates = 10
generations = 5
tournament_size = 3
elitism = True

train_size = 0.2

In [4]:
train_df = raw_train_df.set_index('row_id')

if train_size is None:
    rows = (train_demo_df['adult_child'] == 1) & (train_demo_df['sex'] == 1) & (train_demo_df['handedness'] == 1)
    ideal_subject_ids = train_demo_df.loc[rows].sort_values(by='elbow_to_wrist_cm', ascending=False)['subject'].to_list()

    train_sample_df = train_df.loc[train_df['subject'].isin(ideal_subject_ids), :]
    train_sample_df = train_sample_df[train_sample_df['sequence_type'] == 'Target']
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    print(train_sample_df['sequence_id'].nunique())
else:
    target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

    target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
    target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
    target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

    train_sample_df, test_sample_df = utils.sample_balanced_split(
        target_only_df,
        train_pct=train_size,
        test_pct=0.2
    )

# some_sequences = train_sample_df['sequence_id'].unique()[50:]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

if model_target == 'gesture_action':
    train_sample_df.drop(columns=['gesture_position', 'gesture'], inplace=True)
elif model_target == 'gesture':
    train_sample_df.drop(columns=['gesture_position', 'gesture_action'], inplace=True)
else:
    train_sample_df.drop(columns=['gesture', 'gesture_action'], inplace=True)


Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
pipe_name = 'imu_extractor'
clf_prefix = 'classifier__estimator__'

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

# Fixed values (not tuned)
fixed_params = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__rotation_sensor_list':   [None],
    f'{pipe_name}__sampling_rate':          [100],
    f'{pipe_name}__subject_df':             [train_demo_df],          # not tuned
    f'{pipe_name}__disable_tqdm':           [True],
    f'{pipe_name}__tof_sensor_list':        [None],                   # unused but present
    f'{pipe_name}__band_edges':             [linear_edges],                   # can also be tuned (see below)
    f'{pipe_name}__dc_offset':              [0],
}

if search_mode == 'grid':
    param_grid = {
        **fixed_params,

        # ----- ImuExtractor -----
        f'{pipe_name}__imu_domain':             ['time'],           # FFT or time features
        f'{pipe_name}__rotation_domain':        ['orientation'],          # quaternion‑based or differences
        f'{pipe_name}__thermopile_sensor_list': [thm_columns],      # single or two sensors
        f'{pipe_name}__thermopile_mode':        ['baseline'],
        f'{pipe_name}__tof_mode':               ['baseline'],     # ToF feature extraction mode
        f'{pipe_name}__category_data':          [True],                       # include subject/orientation
        f'{pipe_name}__segmentation':           [None],           # grouping method
        f'{pipe_name}__window':                 [0.5],                    # seconds (only if segmentation='window')
        f'{pipe_name}__step_sec':               [0.2],                    # step (only if window)
        f'{pipe_name}__combine_imu_axes':       [True],
        f'{pipe_name}__combine_rot_axes':       [True],

        # ----- LGBMClassifier -----
        f'{clf_prefix}n_estimators':            [100],
        f'{clf_prefix}max_depth':               [3],
        f'{clf_prefix}learning_rate':           [0.01],
        f'{clf_prefix}subsample':               [0.6],
        f'{clf_prefix}colsample_bytree':        [0.6],
        f'{clf_prefix}min_child_samples':       [10],
        f'{clf_prefix}reg_alpha':               [0.01],
        f'{clf_prefix}reg_lambda':              [0.01],
        f'{clf_prefix}num_leaves':              [15],
        f'{clf_prefix}min_gain_to_split':       [0.0],
    }

elif search_mode == 'random':
    param_grid = {
        **fixed_params,

        f'{pipe_name}__imu_domain':             ['time', 'acceleration', 'velocity', 'displacement'],
        f'{pipe_name}__rotation_domain':        ['orientation', 'motion', 'frequency'],
        f'{pipe_name}__thermopile_sensor_list': [['thm_1'], ['thm_1','thm_2'], ['thm_1','thm_3','thm_5']],
        f'{pipe_name}__thermopile_mode':        ['baseline', 'spatial'],
        f'{pipe_name}__tof_mode':               ['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet'],
        f'{pipe_name}__category_data':          [True, False],
        f'{pipe_name}__segmentation':           [None, 'window', 'phase', 'behavior', 'both'],
        f'{pipe_name}__window':                 uniform(0.2, 2.0),          # 0.2 to 2.2
        f'{pipe_name}__step_sec':               uniform(0.05, 1.0),         # 0.05 to 1.05
        f'{pipe_name}__combine_imu_axes':       [True, False],
        f'{pipe_name}__combine_rot_axes':       [True, False],

        f'{clf_prefix}n_estimators':            randint(50, 1000),
        f'{clf_prefix}max_depth':               randint(2, 15),
        f'{clf_prefix}learning_rate':           loguniform(0.005, 0.5),
        f'{clf_prefix}subsample':               uniform(0.5, 0.5),           # 0.5–1.0
        f'{clf_prefix}colsample_bytree':        uniform(0.5, 0.5),
        f'{clf_prefix}min_child_samples':       randint(5, 200),
        f'{clf_prefix}reg_alpha':               loguniform(1e-3, 10.0),
        f'{clf_prefix}reg_lambda':              loguniform(1e-3, 10.0),
        f'{clf_prefix}num_leaves':              randint(4, 128),
        f'{clf_prefix}min_gain_to_split':       uniform(0.0, 1.0),
    }

elif search_mode == 'bayesian':
    param_grid = {
        # Choices: 0 = None, 1 = use sensor
        f'{pipe_name}__imu_choice':          Categorical([0, 1]),
        f'{pipe_name}__rot_choice':          Categorical([0, 1]),
        f'{pipe_name}__tof_choice':          Categorical([0, 1]),
        f'{pipe_name}__thermo_choice':       Categorical([0, 1]),
        f'{pipe_name}__band_edges_choice':   Categorical([0, 1, 2, 3]),

        f'{pipe_name}__imu_domain':          Categorical(['time', 'acceleration', 'velocity', 'displacement']),
        f'{pipe_name}__rotation_domain':     Categorical(['orientation', 'motion', 'frequency']),
        f'{pipe_name}__thermopile_mode':     Categorical(['baseline', 'spatial']),
        f'{pipe_name}__tof_mode':            Categorical(['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet']),
        f'{pipe_name}__category_data':       Categorical([False]),
        f'{pipe_name}__segmentation':        Categorical(['window']),
        f'{pipe_name}__window':              Real(0.2, 2.0, prior='uniform'),
        f'{pipe_name}__step_sec':            Real(0.05, 1.0, prior='uniform'),
        f'{pipe_name}__combine_imu_axes':    Categorical([True, False]),
        f'{pipe_name}__combine_rot_axes':    Categorical([True, False]),

        f'{clf_prefix}n_estimators':         Integer(50, 1000),
        f'{clf_prefix}max_depth':            Integer(2, 15),
        f'{clf_prefix}learning_rate':        Real(0.005, 0.5, prior='log-uniform'),
        f'{clf_prefix}subsample':            Real(0.5, 1.0),
        f'{clf_prefix}colsample_bytree':     Real(0.5, 1.0),
        f'{clf_prefix}min_child_samples':    Integer(5, 200),
        f'{clf_prefix}reg_alpha':            Real(1e-3, 10.0, prior='log-uniform'),
        f'{clf_prefix}reg_lambda':           Real(1e-3, 10.0, prior='log-uniform'),
        f'{clf_prefix}num_leaves':           Integer(4, 128),
        f'{clf_prefix}min_gain_to_split':    Real(0.0, 1.0),
    }

elif search_mode == 'evolutionary':
    param_grid = {
        # Integer choices (0=None, 1=use)
        f'{pipe_name}__imu_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__rot_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__tof_choice':          GenCategorical([0, 1]),
        f'{pipe_name}__thermo_choice':       GenCategorical([0, 1]),
        f'{pipe_name}__band_edges_choice':   GenCategorical([0, 1, 2, 3]),

        f'{pipe_name}__imu_domain':          GenCategorical(['time', 'acceleration', 'velocity', 'displacement']),
        f'{pipe_name}__rotation_domain':     GenCategorical(['orientation', 'motion', 'frequency']),
        f'{pipe_name}__thermopile_mode':     GenCategorical(['baseline', 'spatial']),
        f'{pipe_name}__tof_mode':            GenCategorical(['baseline', 'spatial', 'edge', 'fft2', 'svd', 'wavelet']),
        f'{pipe_name}__category_data':       GenCategorical([False]),
        f'{pipe_name}__segmentation':        GenCategorical(['window']),
        f'{pipe_name}__window':              Continuous(0.1, 2.0),
        f'{pipe_name}__step_sec':            Continuous(0.01, 1.0),
        f'{pipe_name}__combine_imu_axes':    GenCategorical([True, False]),
        f'{pipe_name}__combine_rot_axes':    GenCategorical([True, False]),

        f'{clf_prefix}n_estimators':         GenInteger(50, 1000),
        f'{clf_prefix}max_depth':            GenInteger(2, 15),
        f'{clf_prefix}learning_rate':        Continuous(0.005, 0.5),
        f'{clf_prefix}subsample':            Continuous(0.5, 1.0),
        f'{clf_prefix}colsample_bytree':     Continuous(0.5, 1.0),
        f'{clf_prefix}min_child_samples':    GenInteger(5, 200),
        f'{clf_prefix}reg_alpha':            Continuous(1e-3, 10.0),
        f'{clf_prefix}reg_lambda':           Continuous(1e-3, 10.0),
        f'{clf_prefix}num_leaves':           GenInteger(4, 128),
        f'{clf_prefix}min_gain_to_split':    Continuous(0.0, 1.0),
    }

In [6]:
if search_mode in ['evolutionary', 'bayesian']:
    custom_extractor = utils.ImuExtractorWithChoice(subject_df=train_demo_df, dc_offset=0)
else:
    custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)
    
lgbm_clf = LGBMClassifier(objective='multiclass', random_state=42, verbose=-1)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=lgbm_clf,
        extractor=custom_extractor,
        mode=None,
        target=model_target
    ))
])

cv = GroupKFold(n_splits=n_splits)

if search_mode == 'grid':
    print('Grid Search')
    search_obj = GridSearchCV(
        pipeline, param_grid, cv=cv, verbose=2, n_jobs=-1, return_train_score=True
    )
elif search_mode == 'random':
    print('Random Search')
    search_obj = RandomizedSearchCV(
        pipeline, param_grid, n_iter=candidates, cv=cv, verbose=2, n_jobs=-1, random_state=42, return_train_score=True
    )
elif search_mode == 'bayesian':
    print('Bayesian Search')
    search_obj = BayesSearchCV(
        pipeline, param_grid, n_iter=candidates, cv=cv, verbose=3, n_jobs=-1, random_state=42, return_train_score=True
    )
elif search_mode == 'evolutionary':
    print('Evolutionary Search')
    search_obj = GASearchCV(
        estimator=pipeline, param_grid=param_grid, cv=n_splits, verbose=3, n_jobs=-1,
        population_size=candidates, generations=generations, tournament_size=tournament_size, elitism=elitism
    )

orientation_cols = ['Lie on Back']

sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols)]

Grid Search


In [7]:
y = sliced_data_df[['sequence_id', model_target]]
groups = sliced_data_df['sequence_id']

if search_mode == 'evolutionary':
    search_obj.fit(sliced_data_df, y)
else:
    search_obj.fit(sliced_data_df, y, groups=groups)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


E0000 00:00:1777462845.781952      59 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777462845.790182      59 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777462845.810349      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810401      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810406      59 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777462845.810409      59 computation_placer.cc:177] computation placer already registered. Please check linka

In [8]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0]
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_lgbm_v2_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")

--- Final Test Results ---
Best CV Score: 0.5239
Test Accuracy: 0.3997

Classification Report:
                precision    recall  f1-score   support

   pinch skin       0.39      0.30      0.34       162
    pull hair       0.55      0.40      0.46       243
pull hairline       0.39      0.15      0.21        81
      scratch       0.32      0.63      0.43       162

     accuracy                           0.40       648
    macro avg       0.41      0.37      0.36       648
 weighted avg       0.43      0.40      0.39       648

Results consolidated and saved to: model_runs/grid_lgbm_v2_results.csv


In [1]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/rnn-methods/cmi_kaggle/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

2026-04-09 21:55:17.025431: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775771717.218177      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775771717.275167      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775771717.740574      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775771717.740613      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775771717.740615      17 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'rnn_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = False

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list': [acc_columns],
    f'{pipe_name}__imu_domain': ['time'],
    f'{pipe_name}__combine_imu_axes': [False],
    f'{pipe_name}__sampling_rate': [100],
    f'{pipe_name}__rotation_sensor_list': [None],
    f'{pipe_name}__thermopile_sensor_list': [None],
    f'{pipe_name}__tof_sensor_list': [None],
    f'{pipe_name}__window': [1.0],
    f'{pipe_name}__step_sec': [0.1],
    f'{pipe_name}__segmentation': ['window'],
    
    'sequence_builder__maxlen': [20],
    'sequence_builder__padding_value': [0.0],
    
    'classifier__estimator__rnn_type': ['rnn','gru', 'lstm'],
    'classifier__estimator__rnn_units': [(64,)],
    'classifier__estimator__dense_units': [(16,)],
    'classifier__estimator__dropout': [0.1],
    'classifier__estimator__learning_rate': [1e-3],
    'classifier__estimator__batch_size': [16],
    'classifier__estimator__epochs': [20],
    'classifier__estimator__patience': [4],
}

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)
sequence_builder = utils.SequencePadder(maxlen=20, padding_value=0.0)
rnn_estimator = utils.KerasRNNClassifier(
    verbose=0
)
classifier = utils.ManyToOneWrapperRNN(
        estimator=rnn_estimator,
        extractor=custom_extractor,
        mode='sequence',
        target='gesture_action',   # or 'gesture'
    )
pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=3)

In [6]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  # LSTM is slow, keep iterations low
                cv=cv,
                random_state=42,
                n_jobs=-1,              # safer with tensorflow/keras on Kaggle
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=-1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
        
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)


Fitting 3 folds for each of 3 candidates, totalling 9 fits


E0000 00:00:1775771766.790033      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775771766.797112      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775771766.815906      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775771766.815949      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775771766.815952      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775771766.815955      56 computation_placer.cc:177] computation placer already registered. Please check linka

In [7]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'lstm_v1_per_gesture_scores.csv')

In [1]:
import os
import warnings
import logging

# ===== SILENCE ALL WARNINGS =====
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress Python warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Suppress TensorFlow logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

# Now import
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/lstm-method/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

# TensorFlow imports with suppressed logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib
from scipy.stats import uniform, loguniform, randint

# Final suppression of retracing warnings (these are annoying but harmless)
tf.keras.backend.clear_session()

E0000 00:00:1775936620.655317      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775936620.717779      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775936621.219954      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220000      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220003      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775936621.220005      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_used = 'gru'

model_run_name = f'{model_used}_v2'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = True
train_size = 0.2
pipe_name = 'temporal_extractor'

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)
num_pattern  = 'acc|rot|thm|tof'

# In your notebook, after defining columns
raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns
)

preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), make_column_selector(pattern="acc|rot|thm|tof")),
], remainder="drop", verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

sequence_builder = utils.SequencePadder(maxlen=60, padding_value=-999.0)  # ← your new window size

rnn_clf = utils.KerasRNNClassifier(
    rnn_type=model_used,
)

classifier = utils.ManyToOneWrapperRNN(estimator=rnn_clf, target="gesture_action")

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ("preprocessor", preprocessor),
    ("sequence_builder", sequence_builder),
    ("classifier", classifier),
])

cv = GroupKFold(n_splits=3)

if random_search:
    param_grid = {
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],
        # Sequence length - capture different gesture durations
        'sequence_builder__maxlen': randint(10, 120),
        'sequence_builder__padding_value': [-999.0],
        
        # RNN architecture size
        'classifier__estimator__rnn_type': [model_used],
        'classifier__estimator__rnn_units': [ (256, 128)],
        'classifier__estimator__dense_units': [(16,),],
        
        # Regularization to match model size
        'classifier__estimator__dropout': uniform(0.4, 0.42),
        'classifier__estimator__learning_rate': loguniform(1e-4, 5e-3),
        
        # Keep these fixed
        'classifier__estimator__bidirectional': [True],
        'classifier__estimator__class_weight_mode': ['balanced'],
        'classifier__estimator__epochs': [100],
        'classifier__estimator__patience': [10],
        'classifier__estimator__batch_size': [16],
    }

else:
    param_grid = {
        # RawSequenceExtractor params
        f'{pipe_name}__acc_cols': [acc_columns],
        f'{pipe_name}__rot_cols': [rot_columns],
        f'{pipe_name}__thm_cols': [thm_columns],
        f'{pipe_name}__tof_cols': [tof_columns],
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],

        # SequencePadder params
        'sequence_builder__maxlen': [120],
        'sequence_builder__padding_value': [-999.0],

        # RNN params (nested under classifier__estimator__)
        'classifier__estimator__rnn_type': [model_used],
        'classifier__estimator__rnn_units': [(128,), (128, 32)],
        'classifier__estimator__dense_units': [(32, 16)],
        'classifier__estimator__dropout': [0.1, 0.14],
        'classifier__estimator__learning_rate': [1e-4],
        'classifier__estimator__batch_size': [16],
        'classifier__estimator__epochs': [120],
        'classifier__estimator__patience': [10],
        "classifier__estimator__bidirectional": [True],
        "classifier__estimator__class_weight_mode": ["balanced"],
    }

In [6]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  
                cv=cv,
                random_state=42,
                n_jobs=1,  
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
    
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


In [7]:
master_cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__estimator__batch_size,param_classifier__estimator__bidirectional,param_classifier__estimator__class_weight_mode,param_classifier__estimator__dense_units,param_classifier__estimator__dropout,param_classifier__estimator__epochs,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,orientation_data,model_target
0,43.201903,10.084561,1.563151,0.055644,16,True,balanced,"(16,)",0.557307,100,...,0.272480,0.079566,5,0.514851,0.39,0.319588,0.408146,0.080742,Lie on Back,gesture_action
1,49.444220,4.762355,1.737553,0.271509,16,True,balanced,"(16,)",0.727470,100,...,0.294774,0.035756,2,0.475248,0.42,0.422680,0.439309,0.025436,Lie on Back,gesture_action
2,53.191242,18.852372,1.898175,0.486233,16,True,balanced,"(16,)",0.441989,100,...,0.285474,0.130655,3,0.742574,0.40,0.824742,0.655772,0.183943,Lie on Back,gesture_action
3,30.490848,2.776558,1.485017,0.042841,16,True,balanced,"(16,)",0.460004,100,...,0.435276,0.097502,1,0.762376,0.75,0.701031,0.737802,0.026488,Lie on Back,gesture_action
4,23.701490,4.376406,1.432013,0.023767,16,True,balanced,"(16,)",0.807362,100,...,0.282487,0.037301,4,0.247525,0.31,0.432990,0.330171,0.077048,Lie on Back,gesture_action


In [ ]:
import os
import warnings
import logging
%pip install -q sktime scikit-optimize sklearn-genetic-opt
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/hot-fix-utils/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Continuous, Categorical as GenCategorical, Integer as GenInteger
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'mini_rocket_v1'
select_k_name = 'select_k_percentile'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
    # 'All': ['Lie on Back', 'Seated Straight', 'Lie on Side - Non Dominant', 'Seated Lean Non Dom - FACE DOWN'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
search_mode  = 'evolutionary'  # Options: 'grid', 'random', 'bayesian', 'evolutionary'
train_size = None

candidates = 10

generations = 5
tournament_size = 3
elitism = True
crossover_probability = 0.8
mutation_probability = 0.1


In [ ]:
train_df = raw_train_df.set_index('row_id')

if train_size is None:
    rows = (train_demo_df['adult_child'] == 1) & (train_demo_df['sex'] == 1) & (train_demo_df['handedness'] == 1)
    ideal_subject_ids = train_demo_df.loc[rows].sort_values(by='elbow_to_wrist_cm', ascending=False)['subject'].to_list()

    train_sample_df = train_df.loc[train_df['subject'].isin(ideal_subject_ids), :]
    train_sample_df = train_sample_df[train_sample_df['sequence_type'] == 'Target']
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    print(train_sample_df['sequence_id'].nunique())
else:
    target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

    target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
    target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
    target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

    train_sample_df, test_sample_df = utils.sample_balanced_split(
        target_only_df,
        train_pct=train_size,
        test_pct=0.2
    )
some_sequences = train_sample_df['sequence_id'].unique()[50:]
train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]



In [ ]:
if search_mode == 'random':
    # ============================================================================
    # RANDOM SEARCH PARAMETER GRID
    # ============================================================================
    param_grid = {
        # ===== RAW SEQUENCE EXTRACTOR =====
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],  # TOF harms performance
        f'{pipe_name}__acc_mode':      ['velocity'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      ['delta'],
        f'{pipe_name}__tof_mode':      ['baseline'],
        
        # ===== SEQUENCE BUILDER =====
        'sequence_builder__maxlen':         randint(30, 70),  # Shorter = less overfitting
        'sequence_builder__padding_value':  [-999.0],
        
        # ===== MINIROCKET CORE =====
        'classifier__estimator__num_kernels': randint(300, 1500),  # Fewer = more regularization
        'classifier__estimator__random_state': [42],
        
        # ===== CLASSIFIER TYPE SELECTION =====
        'classifier__estimator__classifier_type': [
            'poisson', 'tweedie',  # Poisson-based models
            'logistic', 'sgd', 'passive',        # Linear classifiers
            'ridge', 'svm',          # Ridge & SVM
            'mlp'                                 # Neural network
        ],
        
        # ===== COMMON PARAMETERS =====
        'classifier__estimator__class_weight': ['balanced'],  # Handle imbalanced data
        'classifier__estimator__max_iter': [1000],
        
        # ===== LOGISTIC REGRESSION PARAMETERS =====
        # C: Inverse regularization (smaller = stronger regularization)
        'classifier__estimator__C': loguniform(0.001, 100),  # For logistic, passive, svm
        'classifier__estimator__penalty': ['l2'],  # L2 regularization
        'classifier__estimator__solver': ['lbfgs', 'saga'],  # lbfgs for l2, saga for elasticnet
        
        # ===== SGD CLASSIFIER PARAMETERS =====
        # alpha: Regularization strength (larger = stronger)
        'classifier__estimator__alpha': loguniform(1e-6, 0.1),  # For sgd, ridge, mlp, poisson, tweedie
        'classifier__estimator__loss': ['log_loss', 'hinge'],  # log_loss for probabilities, hinge for SVM-style
        'classifier__estimator__learning_rate': ['optimal', 'adaptive'],
        
        # ===== PASSIVE AGGRESSIVE PARAMETERS =====
        # C_passive: Regularization (smaller = stronger)
        'classifier__estimator__C_passive': loguniform(0.001, 10),
        
        # ===== RIDGE CLASSIFIER PARAMETERS =====
        # alpha_ridge: Regularization (larger = stronger)
        'classifier__estimator__alpha_ridge': loguniform(0.01, 100),
        
        # ===== SVM PARAMETERS =====
        # C_svm: Inverse regularization (smaller = stronger)
        'classifier__estimator__C_svm': loguniform(0.001, 10),
        'classifier__estimator__loss_svm': ['squared_hinge', 'hinge'],
        
        # ===== MLP PARAMETERS =====
        'classifier__estimator__hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
        'classifier__estimator__activation': ['relu', 'tanh'],
        'classifier__estimator__alpha_mlp': loguniform(1e-5, 0.01),  # L2 regularization
        'classifier__estimator__learning_rate_init': loguniform(0.0001, 0.01),
        'classifier__estimator__early_stopping': [True],  # Built-in regularization
        
        # ===== POISSON REGRESSION PARAMETERS =====
        # For classifier_type='poisson' only
        'classifier__estimator__alpha_poisson': loguniform(0.01, 10),  # Regularization
        
        # ===== TWEEDIE REGRESSION PARAMETERS =====
        # For classifier_type='tweedie' only
        'classifier__estimator__power': [1.0, 1.5, 2.0],  # 1=Poisson, 1.5=NegBinom, 2=Gamma
        'classifier__estimator__alpha_tweedie': loguniform(0.01, 10),
    }

elif search_mode == 'bayesian':
    param_grid = {
        f'{pipe_name}__acc_choice':         Categorical([0, 1]),
        f'{pipe_name}__rot_choice':         Categorical([0, 1]),
        f'{pipe_name}__thm_choice':         Categorical([0, 1]),
        f'{pipe_name}__tof_choice':         Categorical([0, 1]),
        f'{pipe_name}__acc_mode':           Categorical(['raw', 'velocity', 'displacement']),
        f'{pipe_name}__rotation_mode':      Categorical(['quaternion', 'euler', 'delta_euler']),
        f'{pipe_name}__thm_mode':           Categorical(['raw', 'delta', 'centered', 'average']),
        f'{pipe_name}__tof_mode':           Categorical(['raw', 'delta', 'centered', 'baseline']),
        'sequence_builder__maxlen':         Integer(5, 100),
        'classifier__estimator__num_kernels': Integer(84, 1500),
        'classifier__estimator__classifier_type': Categorical(['logistic']),
        'classifier__estimator__C':         Real(0.001, 100.0, prior='log-uniform'),
        'classifier__estimator__alpha':     Real(1e-6, 0.1, prior='log-uniform'),
        'feature_selection__percentile':    Integer(5, 100),
    }

elif search_mode == 'evolutionary':
    param_grid = {
        # Sensor choices (integers)
        f'{pipe_name}__acc_choice':         GenCategorical([0, 1]),
        f'{pipe_name}__rot_choice':         GenCategorical([0, 1]),
        f'{pipe_name}__thm_choice':         GenCategorical([0, 1]),
        f'{pipe_name}__tof_choice':         GenCategorical([0, 1]),
        
        # Sensor modes (strings)
        f'{pipe_name}__acc_mode':           GenCategorical(['raw', 'velocity', 'displacement', 'smoothed']),
        f'{pipe_name}__rotation_mode':      GenCategorical(['quaternion', 'euler', 'delta_euler']),
        f'{pipe_name}__thm_mode':           GenCategorical(['raw', 'delta', 'centered', 'average']),
        f'{pipe_name}__tof_mode':           GenCategorical(['raw', 'delta', 'centered', 'baseline']),
        
        # Sequence & Rocket (integers/floats)
        'sequence_builder__maxlen':                 GenInteger(5, 100),
        'classifier__estimator__num_kernels':       GenInteger(84, 1500),
        'classifier__estimator__classifier_type':   GenCategorical(['logistic']),
        'classifier__estimator__C':                 Continuous(0.001, 100.0),
        'classifier__estimator__alpha':             Continuous(1e-6, 0.1),
        'feature_selection__percentile':            GenInteger(5, 100),
    }

else: # 'grid' mode
    # ============================================================================
    # GRID SEARCH PARAMETER GRID (Full factorial)
    # ============================================================================
    param_grid = {
        # ===== RAW SEQUENCE EXTRACTOR =====
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],  # TOF harms performance - keep disabled
        f'{pipe_name}__acc_mode':      ['velocity'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      ['delta'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        'feature_selection__percentile': [10],
        
        # ===== SEQUENCE BUILDER =====
        'sequence_builder__maxlen':         [55],  # Test different sequence lengths
        'sequence_builder__padding_value':  [-999.0],
        
        # ===== MINIROCKET CORE =====
        'classifier__estimator__num_kernels': [50],  # Fewer = more regularization
        'classifier__estimator__random_state': [42],
        
        # ===== CLASSIFIER TYPE SELECTION =====
        'classifier__estimator__classifier_type': [
            'logistic',  # Start with logistic regression (best balance)
            # 'poisson',  # Uncomment for count data
            # 'poisson_nb',  # Uncomment for classification with count features
            # 'tweedie',  # Uncomment for overdispersed count data
            # 'sgd',  # Uncomment for large datasets
            # 'ridge',  # Uncomment for fast training
            # 'svm',  # Uncomment for max-margin classification
            # 'mlp',  # Uncomment for non-linear patterns (slow)
        ],
        
        # # ===== COMMON PARAMETERS =====
        'classifier__estimator__class_weight': ['balanced'],  # Handle imbalanced data
        'classifier__estimator__max_iter': [1000],
        
        # ===== LOGISTIC REGRESSION (classifier_type='logistic') =====
        # C: Inverse regularization (0.01=strong, 0.1=moderate, 1=weak, 10=minimal)
        'classifier__estimator__C': [1e-10],
        'classifier__estimator__penalty': ['l2'],  # L2 regularization only
        'classifier__estimator__solver': ['saga'],  # Good for L2 penalty
        
        # ===== SGD CLASSIFIER (classifier_type='sgd') =====
        # alpha: Regularization strength (0.0001=weak, 0.001=moderate, 0.01=strong)
        'classifier__estimator__alpha': [0.01],
        'classifier__estimator__loss': ['log_loss'],  # log_loss for probabilities
        'classifier__estimator__penalty_sgd': ['l2'],  # L2 regularization
        
        # ===== PASSIVE AGGRESSIVE (classifier_type='passive') =====
        # C_passive: Regularization (0.01=strong, 0.1=moderate, 1=weak)
        'classifier__estimator__C_passive': [0.01],
        
        # ===== RIDGE CLASSIFIER (classifier_type='ridge') =====
        # alpha_ridge: Regularization (0.1=weak, 1=moderate, 10=strong, 100=very strong)
        'classifier__estimator__alpha_ridge': [500.0],
        
        # ===== SVM (classifier_type='svm') =====
        # C_svm: Inverse regularization (0.01=strong, 0.1=moderate, 1=weak)
        'classifier__estimator__C_svm': [0.01],
        'classifier__estimator__loss_svm': ['squared_hinge'],  # Faster than hinge
        
        # ===== MLP CLASSIFIER (classifier_type='mlp') =====
        # hidden_layer_sizes: Architecture (smaller = less overfitting)
        'classifier__estimator__hidden_layer_sizes': [(50,)],
        'classifier__estimator__activation': ['relu'],  # Standard choice
        'classifier__estimator__alpha_mlp': [0.0001],  # L2 regularization
        'classifier__estimator__learning_rate_init': [0.001],
        'classifier__estimator__early_stopping': [True],  # Prevents overfitting
        
        # ===== POISSON REGRESSION (classifier_type='poisson') =====
        # For count data where variance = mean
        'classifier__estimator__alpha_poisson': [1.0],
        
        # ===== POISSON NAIVE BAYES (classifier_type='poisson_nb') =====
        # For classification with count features (e.g., word counts, sensor readings)
        'classifier__estimator__alpha_nb': [1.0],  # Laplace smoothing
        
        # ===== TWEEDIE REGRESSION (classifier_type='tweedie') =====
        # For overdispersed count data (variance > mean)
        'classifier__estimator__power': [1.0],  # 1=Poisson, 1.5=NegBinom, 2=Gamma
        'classifier__estimator__alpha_tweedie': [1.0]
    }

In [ ]:
importlib.reload(utils)

if search_mode in ['bayesian', 'evolutionary']:
    raw_extractor = utils.RawSequenceExtractorWithChoice()
else:
    raw_extractor = utils.RawSequenceExtractor(
        acc_cols=acc_columns,
    )   

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=110, padding_value=-999.0)

# IMPORTANT: Use create_rocket_classifier with default params
# The grid search will override them
rocket_clf = utils.create_rocket_classifier(
    classifier_type='ridge',  # Default, will be overridden by grid search
    num_kernels=1000,
    random_state=42
)

classifier = utils.ManyToOneWrapperRNN(estimator=rocket_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('feature_selection', utils.RocketFeatureSelector()),
    ('classifier', classifier),
])

group_kfold = GroupKFold(n_splits=n_splits)

In [ ]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if search_mode == 'random':
            print('random search')
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=candidates,
                cv=group_kfold,
                random_state=42,
                n_jobs=1,
                verbose=3,
                return_train_score=True
            )
        elif search_mode == 'bayesian':
            print('bayesian search')
            search_obj = BayesSearchCV(
                estimator=pipeline,
                search_spaces=param_grid,
                n_iter=candidates,  
                cv=group_kfold,
                random_state=42,
                n_jobs=-1,
                verbose=3,
                return_train_score=True
            )
        elif search_mode == 'evolutionary':
            print('evolutionary search')
            search_obj = GASearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=n_splits,
                verbose=3,
                n_jobs=1,
                population_size=candidates,
                generations=generations,
                tournament_size=tournament_size,
                elitism=elitism,
                crossover_probability=crossover_probability,
                mutation_probability=mutation_probability
            )
        else:
            print('grid search')
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=group_kfold,
                verbose=3,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        if search_mode == 'evolutionary':
            search_obj.fit(sliced_data_df, y)
        else:
            search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['train_size'] = train_size
    master_cv_results_df['search_mode'] = search_mode
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

In [ ]:
import pandas as pd
import numpy as np
import os
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib
import warnings

from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow import keras

import utils

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
sampling_rate = 100
n_splits = 3
pipe_name = 'imu_extractor'

acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165']

model_run_name = 'neural_network.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

In [ ]:
train_df = raw_train_df.set_index('row_id')
multiple_gesture_df = train_df[train_df['sequence_id'].isin(['SEQ_000007', 'SEQ_000008', 'SEQ_000013'])]
train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.6, test_pct=0.2)

In [ ]:
num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x', 'acc_y', 'acc_z'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z']
)

nn_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    learning_rate_init=0.001,
    max_iter=200,
    validation_fraction=0.1,
    random_state=42,
    verbose=False
)

In [ ]:
importlib.reload(utils)

sliced_df = train_sample_df.copy(deep=True)

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [None],
    f'{pipe_name}__combine_rot_axes': [True],

    f'{pipe_name}__thermopile_sensor_list': [None],
    f'{pipe_name}__tof_sensor_list': [None],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],
    f'{pipe_name}__category_data': [True],
    f'{pipe_name}__segmentation': ['phase', 'window'],

    'classifier__estimator__hidden_layer_sizes': [(32,)],
    'classifier__estimator__activation': ['relu'],
    'classifier__estimator__learning_rate_init': [0.001],
    'classifier__estimator__max_iter': [200],
    'classifier__estimator__alpha': [1], 
}

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(nn_clf, custom_extractor, mode=None))
])


y = sliced_df[['sequence_id', 'gesture']]
groups = sliced_df['sequence_id']

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

grid_search.fit(sliced_df, y, groups=groups)
pbar.close()

In [ ]:
grid_search.cv_results_

In [ ]:
pd.DataFrame(grid_search.cv_results_).T

In [ ]:
# Permutation Importance
from sklearn.inspection import permutation_importance

# Get best model
best_model = grid_search.best_estimator_

# Transform sliced_df (same data used in grid search)
X_transformed = best_model.named_steps['imu_extractor'].transform(sliced_df)
X_preprocessed = best_model.named_steps['preprocessor'].transform(X_transformed)

# Get labels
y_seq = sliced_df.groupby('sequence_id')['gesture'].first()
y_aligned = y_seq.reindex(X_preprocessed.index)

# Permutation importance
perm_importance = permutation_importance(
    best_model.named_steps['classifier'].estimator_,
    X_preprocessed, y_aligned,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm_importance.importances_mean
}).sort_values('importance', ascending=False)

# Plot top 20
plt.figure(figsize=(10, 8))
plt.barh(feat_imp_df['feature'].head(20), feat_imp_df['importance'].head(20))
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Permutation)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Save
feat_imp_df.to_csv(model_run_folder_name + 'neural_network_feature_importance.csv', index=False)
print(feat_imp_df.head(20))

In [ ]:
model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'neural_network_results.csv', index=False)

print(f"Best score: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
import importlib
import joblib

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
eg_subject = 'SUBJ_040724'
single_sequence = 'SEQ_065189'
dc_offset_max = 10
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
log_edges = np.logspace(np.log10(0.5), np.log10(100), num=15)
sequences = ['SEQ_000034', 'SEQ_065478', 'SEQ_065470']
subjects = ['SUBJ_059520']
band_edges = np.arange(0, 101, 10)
pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
subject_cols = ['orientation', 'subject', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000016', 'SEQ_000018',
                    'SEQ_000022', 'SEQ_000033', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053',
                    'SEQ_000058', 'SEQ_000063', 'SEQ_000079', 'SEQ_000091', 'SEQ_000092',
                    'SEQ_000111', 'SEQ_000113', 'SEQ_000114', 'SEQ_000142', 'SEQ_000150']
some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165',
                'SUBJ_024086', 'SUBJ_040733', 'SUBJ_063346', 'SUBJ_055211', 'SUBJ_001430',
                'SUBJ_012088', 'SUBJ_040310', 'SUBJ_032233', 'SUBJ_059330', 'SUBJ_013623',
                'SUBJ_032585', 'SUBJ_063464', 'SUBJ_038023', 'SUBJ_044680', 'SUBJ_024137']

model_run_name = 'all_domains.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

train_sample_pct = 0.20
test_sample_pct = 0.05

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

In [ ]:
train_df = raw_train_df.set_index('row_id')

single_subject_df = train_df.loc[train_df['subject'] == eg_subject, :]

single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

multiple_gesture_df = train_df.loc[train_df['sequence_id'].isin(some_sequences), :]

some_subjects_df = train_df.loc[train_df['subject'].isin(some_subjects), :]

master_test_df = raw_test_df.merge(test_demo_df, on='subject', how='left').set_index('row_id')

train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.2, test_pct=0.2)
train_sample_df['sequence_id'].nunique(), train_sample_df.shape

In [ ]:
# 0.47 across all domains

importlib.reload(utils)

num_pattern = 'acc|rot|thm'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation', 'subject']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = single_subject_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            OrdinalEncoder(),
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z']
)

preprocessor.set_output(transform='pandas')

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier(), custom_extractor))
])

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['acceleration'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [['rot_w','rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes': [True],
    f'{pipe_name}__rotation_domain': ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],

    f'{pipe_name}__tof_sensor_list': [None],

    f'{pipe_name}__window': [0.3],
    f'{pipe_name}__step_sec': [0.05],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],

    f'{pipe_name}__category_data': [True],
    
    f'{pipe_name}__segmentation': [None]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    verbose=1,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', 'gesture']]
n_fits = len(list(ParameterGrid(param_grid))) * n_splits

with utils.tqdm_joblib(tqdm(total=n_fits, desc="Grid Search Fits")):
    grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)

In [ ]:
cv_results_df[['param_imu_extractor__sampling_rate', 'param_imu_extractor__band_edges', 'param_imu_extractor__segmentation', 'mean_test_score', 'rank_test_score']].sort_values(by='rank_test_score')

In [ ]:
best_model = grid_search.best_estimator_

y_test = test_sample_df[['sequence_id', 'gesture']]

score = best_model.score(test_sample_df, y_test)
print(f"Test score: {score:.4f}")

In [ ]:
# Get best model
best_model = grid_search.best_estimator_

# Get feature names from preprocessor
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# Get classifier and importances
rf = best_model.named_steps['classifier'].estimator_
importances = rf.feature_importances_

# Create DataFrame
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feat_imp_df['items'] = feat_imp_df['feature'].str.split('_')
feat_imp_df['size_items'] = feat_imp_df['items'].apply(len)
feat_imp_df['name'] = feat_imp_df['items'].str[0]

feat_imp_df['percentage'] = feat_imp_df['importance'] / feat_imp_df['importance'].sum() * 100
feat_imp_df['created_feature'] = feat_imp_df['items'].str[-1]

feat_imp_df.groupby(['name', 'created_feature'])['percentage'].sum().sort_values(ascending=False)


In [ ]:
feat_imp_df.groupby('created_feature')['percentage'].sum().sort_values(ascending=False)

In [ ]:
# Plot top 20
plt.figure(figsize=(10, 8))
plt.barh(feat_imp_df['feature'].head(20), feat_imp_df['percentage'].head(20))
plt.xlabel('Percentage of Total Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
eg = single_gesture_df.drop(columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 
                                  'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 
                                  'acc_z', 'rot_w','rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'])[:1].T.reset_index()
eg['sensor'] = eg['index'].str.split('_').str[1]
eg['node'] = eg['index'].str.split('_').str[2].str.split('v').str[1].astype(int)
eg

In [ ]:
importlib.reload(utils)

extractor = utils.ImuExtractor(
                imu_sensor_list=['acc_x'],
                category_data=True,
                segmentation='both'
                                   )

pipeline = Pipeline([
    (pipe_name, utils.ImuExtractor(
                                #    imu_sensor_list=['acc_x'],
                              #   rotation_sensor_list=['rot_x', 'rot_y'],
                            #   thermopile_sensor_list=['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'],
                                   category_data=True,
                                   segmentation=None,
                                # segmentation=None,
                                combine_rot_axes = True,
                                window = 0.3,
                                step_sec=0.05,
                                tof_sensor_list=tof_columns
                                   )),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier(), extractor))
])
pipeline['preprocessor'].set_output(transform='pandas')

X_eg = pipeline[:1].fit_transform(single_gesture_df)
y_eg = single_gesture_df[['sequence_id', 'gesture']]
X_eg

In [ ]:
segment_lengths = (
    train_sample_df.groupby(['sequence_id', 'phase'])['sequence_counter']
    .count()
    .reset_index()
    .rename(columns={'sequence_counter': 'length_samples'})
)

segment_lengths['length_sec'] = segment_lengths['length_samples'] / sampling_rate

print(segment_lengths['length_sec'].describe())
segment_lengths['length_sec'].hist(bins=50)

In [ ]:
segment_lengths_beh = (
    train_sample_df.groupby(['sequence_id', 'behavior'])['sequence_counter']
    .count()
    .reset_index()
    .rename(columns={'sequence_counter': 'length_samples'})
)
segment_lengths_beh['length_sec'] = segment_lengths_beh['length_samples'] / sampling_rate
print(segment_lengths_beh['length_sec'].describe())
segment_lengths_beh['length_sec'].hist(bins=50)

In [ ]:
# Phase breakdown
phase_stats = (
    train_sample_df.groupby(['sequence_id', 'phase'])['sequence_counter']
    .count()
    .reset_index()
    .rename(columns={'sequence_counter': 'length_samples'})
)
phase_stats['length_sec'] = phase_stats['length_samples'] / sampling_rate
print("=== PHASE ===")
print(phase_stats.groupby('phase')['length_sec'].describe().round(2))

# Behavior breakdown
beh_stats = (
    train_sample_df.groupby(['sequence_id', 'behavior'])['sequence_counter']
    .count()
    .reset_index()
    .rename(columns={'sequence_counter': 'length_samples'})
)
beh_stats['length_sec'] = beh_stats['length_samples'] / sampling_rate
print("\n=== BEHAVIOR ===")
print(beh_stats.groupby('behavior')['length_sec'].describe().round(2))

In [ ]:
import os
import warnings
import importlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from itertools import combinations
import utils

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
def find_data_root(local_dir='data', kaggle_root='/kaggle/input'):
    local_path = Path(local_dir)
    required = [
        'train.csv',
        'test.csv',
        'train_demographics.csv',
        'test_demographics.csv'
    ]

    if local_path.exists() and all((local_path / f).exists() for f in required):
        print(f'Using local data folder: {local_path.resolve()}')
        return local_path

    kaggle_root = Path(kaggle_root)
    if kaggle_root.exists():
        for csv_path in kaggle_root.rglob('train.csv'):
            candidate = csv_path.parent
            if all((candidate / f).exists() for f in required):
                print(f'Using Kaggle data folder: {candidate}')
                return candidate

    raise FileNotFoundError(
        'Could not find the dataset locally or in /kaggle/input. '
        'Place the CSV files in ./data/ or attach the Kaggle dataset.'
    )

data_folder = find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'svm_rbf_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

path_to_model_run_name = model_run_folder_name + model_run_name

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

model_target = 'gesture_action'

model_run_name = 'svm_rbf_v1'

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

# target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

some_sequences = train_sample_df['sequence_id'].unique()[:50]
train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_sample_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain':             ['acceleration'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [None],
    f'{pipe_name}__combine_rot_axes':       [True],
    f'{pipe_name}__rotation_domain':        ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [None],
    f'{pipe_name}__tof_sensor_list':        [None],
    f'{pipe_name}__tof_mode':               ['research'],

    f'{pipe_name}__window':                 [0.2],
    f'{pipe_name}__step_sec':               [0.037],
    f'{pipe_name}__dc_offset':              [1.5],
    f'{pipe_name}__band_edges':             [None],
    f'{pipe_name}__category_data':          [True],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [0.8], 

    'classifier__estimator__kernel':        ['rbf'],
    'classifier__estimator__C':             [3],
    'classifier__estimator__gamma':         ['scale'],
    'classifier__estimator__class_weight':  ['balanced'],
}

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

svm_clf = SVC(
    kernel='rbf',
    decision_function_shape='ovr',
    probability=False,
    random_state=42
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('pca', utils.IndexPreservingPCA()),
    ('classifier', utils.ManyToOneWrapper(
        estimator=svm_clf,
        extractor=custom_extractor,
        mode=None,
        target=model_target
    ))
])

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    verbose=2,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', model_target]]

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

model = grid_search.best_estimator_
joblib.dump(model, path_to_model_run_name)

In [ ]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + f'{model_run_name}_results.csv', index=False)

grid_search.best_score_

In [ ]:
cv_df = cv_results_df.sort_values('rank_test_score').T.reset_index()

split_frame_series = cv_df['index'].str.replace('__', '_').str.split('_')
imu_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'imu')
classifier_rows = (split_frame_series.str[0] == 'param') & (split_frame_series.str[1] == 'classifier')

cv_df.loc[split_frame_series.str[-1] == 'time', 'name'] = split_frame_series.str[0:2].str.join('_')
cv_df.loc[split_frame_series.str[-1] == 'time', 'eval'] = 'time'

cv_df.loc[classifier_rows, 'eval'] = 'classifier'
cv_df.loc[classifier_rows, 'name'] = split_frame_series.loc[classifier_rows].str[3:].str.join('_')

cv_df.loc[imu_rows, 'eval'] = 'imu'
cv_df.loc[imu_rows, 'name'] = split_frame_series.loc[imu_rows].str[3:].str.join('_')

cv_df.loc[split_frame_series.str[0] == 'mean', 'eval'] = 'result'
cv_df.loc[split_frame_series.str[0] == 'mean', 'name'] = split_frame_series.str[0:2].str.join('_')

cv_df.loc[split_frame_series.str[0] == 'std', 'eval'] = 'result'
cv_df.loc[split_frame_series.str[0] == 'std', 'name'] = split_frame_series.str[0:2].str.join('_')

cv_df = cv_df.dropna(subset=['eval', 'name'])
cv_df = cv_df[['eval', 'name', 0]]
cv_df.columns = ['eval', 'name', 'value']

display(cv_df[cv_df['eval'] == 'classifier'].reset_index(drop=True))
display(cv_df[cv_df['eval'] == 'result'].reset_index(drop=True))

In [ ]:
best_model = grid_search.best_estimator_

extractor    = best_model.named_steps['imu_extractor']
preprocessor = best_model.named_steps['preprocessor']
classifier   = best_model.named_steps['classifier']

X_feat = extractor.transform(test_sample_df)
X_proc = preprocessor.transform(X_feat)

y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
y_true = y_true.reindex(X_proc.index)

y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

print(classification_report(y_true, y_pred))

report_df = pd.DataFrame(
    classification_report(y_true, y_pred, output_dict=True)
).T.sort_values('f1-score', ascending=False)

report_df.to_csv(model_run_folder_name + 'svm_rbf_per_gesture_scores.csv')
report_df

In [ ]:
group_dict = {
    'acc': acc_columns,
    'rot': rot_columns,
    'thm': thm_columns,
    'tof': tof_columns,
    # 'misc': ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm', 'adult_child', 'sex', 'handedness', 'segment_id']
}

all_combos = {}

for r in range(1, len(group_dict) + 1):
    for keys in combinations(group_dict.keys(), r):
        name = '_'.join(keys)
        cols = sum([group_dict[k] for k in keys], [])
        all_combos[name] = cols

threshold = 0.8

fig, ax = plt.subplots(figsize=(14, 10), nrows=5, ncols=3)

for i, (key, value) in enumerate(all_combos.items()):
    X = train_sample_df[value]
    X_processed = preprocessor.fit_transform(X).fillna(0)
    pca = PCA()
    pca.fit(X_processed)
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    ax[i // 3, i % 3].plot(cumulative_variance, marker='o')
    ax[i // 3, i % 3].set_title(key)
    ax[i // 3, i % 3].grid(True)

    # Find where cumulative variance reaches >= 0.8
    idx_80 = np.where(cumulative_variance >= threshold)[0]
    if len(idx_80) > 0:
        comp_num = idx_80[0] + 1  # +1 because components start at 1
        ax[i // 3, i % 3].axvline(x=comp_num, color='red', linestyle='--', alpha=0.7)
        ax[i // 3, i % 3].text(comp_num + 0.5, threshold + 0.02, 
                                 f'{comp_num} comps', color='red', fontsize=8)

fig.supxlabel('Number of Components')
fig.supylabel('Cumulative Explained Variance')
plt.tight_layout()
plt.show()

In [1]:
import os
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/tcn-script/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

E0000 00:00:1775939113.008366      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775939113.068629      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775939113.587984      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775939113.588025      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775939113.588029      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775939113.588032      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'tcn_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False
train_size = 0.8

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)

Train: 3856 seqs | 75.4%
Test:  629 seqs  | 12.3%


In [5]:
importlib.reload(utils)

raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns
)

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=80, padding_value=-999.0)

tcn_clf = utils.KerasTCNClassifier()

classifier = utils.ManyToOneWrapperRNN(estimator=tcn_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=n_splits)

In [6]:
if random_search:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode':      ['centered'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        'sequence_builder__maxlen':         [120],
        'sequence_builder__padding_value':  [-999.0],

        # ============ TCN ============
        'classifier__estimator__nb_filters': [64],
        'classifier__estimator__kernel_size': [3],  # Uniform integer 2-8
        'classifier__estimator__nb_stacks': [1],  # Uniform integer 1-4
        'classifier__estimator__dilations': [(1, 2, 4, 8, 16, 32, 64)],
        'classifier__estimator__use_skip_connections': [True],
        'classifier__estimator__dense_units': [(64,)],
        'classifier__estimator__dropout': [0.14],  # Uniform 0.1-0.5
        'classifier__estimator__learning_rate': loguniform(0.0016, 0.002),  # Log uniform
        'classifier__estimator__batch_size': [32],  # Uniform 8-64
        'classifier__estimator__patience': [10],  # Uniform 10-20
    }
else:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode':      ['centered'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        'sequence_builder__maxlen':         [110],
        'sequence_builder__padding_value':  [-999.0],

        # ============ TCN ============
        'classifier__estimator__nb_filters': [64],  # Uniform integer
        'classifier__estimator__kernel_size': [3],  # Uniform integer 2-8
        'classifier__estimator__nb_stacks': [1],  # Uniform integer 1-4
        'classifier__estimator__dilations': [(1, 2, 4, 8, 16, 32, 64)],
        'classifier__estimator__use_skip_connections': [True],
        'classifier__estimator__dense_units': [(64,)],
        'classifier__estimator__dropout': [0.14],  # Uniform 0.1-0.5
        'classifier__estimator__learning_rate': [0.0018],  # Log uniform
        'classifier__estimator__batch_size': [32],  # Uniform 8-64
        'classifier__estimator__patience': [10],  # Uniform 10-20
        'classifier__estimator__class_weight_mode':   ['balanced'],
    }

In [7]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=2,
                cv=cv,
                random_state=42,
                n_jobs=1,
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['train_size'] = train_size
    master_cv_results_df['is_random_search'] = random_search
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


I0000 00:00:1775939170.099450      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775939170.105387      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775939183.204432      72 service.cc:152] XLA service 0x796584038d90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775939183.204470      72 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775939183.204492      72 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775939185.385118      72 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775939201.863499      72 device_compiler.h:188] Compiled clust

In [8]:
master_cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__estimator__batch_size,param_classifier__estimator__class_weight_mode,param_classifier__estimator__dense_units,param_classifier__estimator__dilations,param_classifier__estimator__dropout,param_classifier__estimator__kernel_size,...,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,orientation_data,model_target,train_size,is_random_search
0,63.332112,3.487586,7.92191,0.109657,32,balanced,"(64,)","(1, 2, 4, 8, 16, 32, 64)",0.14,3,...,1,0.656051,0.713831,0.686804,0.685562,0.023605,Lie on Back,gesture_action,0.8,False


In [ ]:
import pandas as pd
import utils
import os

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]
target_only_df.head()

In [26]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
import os
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/cmi-experiment-utilities/cmi_kaggle/')
sys.path.append('/kaggle/input/cmi-competition-code')
import visual_utils_interactive as visual_utils
import utils
import matplotlib.pyplot as plt

In [28]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


PosixPath('/kaggle/input/competitions/cmi-detect-behavior-with-sensor-data')

In [29]:
# Define sensor columns for clustering
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 
               'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

nan_tof_rows = raw_train_df[tof_columns].isna().any(axis=1)
nan_sequence_ids = raw_train_df.loc[nan_tof_rows, 'sequence_id'].unique()
clean_df = raw_train_df[~raw_train_df['sequence_id'].isin(nan_sequence_ids)].set_index('row_id')
clean_df['gesture_position'] = clean_df['gesture'].str.split(' - ').str[0]
clean_df['gesture_action'] = clean_df['gesture'].str.split(' - ').str[-1]

In [30]:
explorer = visual_utils.DataExplorerApp(clean_df, train_demo_df)
explorer.show()

HTML(value="<h2 style='margin:8px 0'>🎮 IMU Data Explorer</h2>")

HTML(value='<div style="border:2px solid #dee2e6;border-radius:8px;padding:10px;background:#f8f9fa;font-family…

Output()

In [ ]:
sample_sequence_df = clean_df[clean_df['sequence_id'] == 'SEQ_065522']

dummy_extractor = utils.ImuExtractor(
    imu_sensor_list=['acc_x'],
    imu_domain='acceleration'
    )

fig, ax = plt.subplots(figsize=(10, 6), nrows = 2, ncols = 2)
ax=ax.ravel()

sample_sequence_df[dummy_extractor.imu_sensor_list].reset_index(drop=True).plot(ax=ax[0])
dummy_extractor.process_for_imu_values(sample_sequence_df).plot(ax=ax[1])
utils.ImuExtractor(imu_sensor_list=['acc_x'], imu_domain='velocity').process_for_imu_values(sample_sequence_df).plot(ax=ax[2])
utils.ImuExtractor(imu_sensor_list=['acc_x'], imu_domain='displacement').process_for_imu_values(sample_sequence_df).plot(ax=ax[3])

In [ ]:
# Check your acceleration range
print(f"Acceleration min: {sample_sequence_df['acc_x'].min():.2f}")
print(f"Acceleration max: {sample_sequence_df['acc_x'].max():.2f}")
print(f"Acceleration mean: {sample_sequence_df['acc_x'].mean():.2f}")

In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
import importlib
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore', message='Mean of empty slice')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000016', 'SEQ_000018',
                    'SEQ_000022', 'SEQ_000033', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053',
                    'SEQ_000058', 'SEQ_000063', 'SEQ_000079', 'SEQ_000091', 'SEQ_000092',
                    'SEQ_000111', 'SEQ_000113', 'SEQ_000114', 'SEQ_000142', 'SEQ_000150']
some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165',
                'SUBJ_024086', 'SUBJ_040733', 'SUBJ_063346', 'SUBJ_055211', 'SUBJ_001430',
                'SUBJ_012088', 'SUBJ_040310', 'SUBJ_032233', 'SUBJ_059330', 'SUBJ_013623',
                'SUBJ_032585', 'SUBJ_063464', 'SUBJ_038023', 'SUBJ_044680', 'SUBJ_024137']

model_run_name = 'xgboost.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

train_sample_pct = 0.20
test_sample_pct = 0.05

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

In [ ]:
train_df = raw_train_df.set_index('row_id')

some_subjects_df = train_df[train_df['subject'].isin(some_subjects)]

multiple_gesture_df = train_df[train_df['sequence_id'].isin(['SEQ_065471', 'SEQ_065478', 'SEQ_065487'])]

train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.2, test_pct=0.2)

In [ ]:
# 0.47 across all domains

importlib.reload(utils)

num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = multiple_gesture_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list': [[None], ['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['time', 'velocity', 'frequency', 'acceleration'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [[None], ['rot_w','rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes': [True],
    f'{pipe_name}__rotation_domain': ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [[None], ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],

    f'{pipe_name}__tof_sensor_list': [tof_columns],
    f'{pipe_name}__tof_mode': ['simple', 'research'],

    f'{pipe_name}__window': [0.25],
    f'{pipe_name}__step_sec': [0.1],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],

    f'{pipe_name}__category_data': [True],
    
    f'{pipe_name}__segmentation': [None, 'window' 'phase'],

    'classifier__estimator__n_estimators': [200],
    'classifier__estimator__max_depth': [4],
    'classifier__estimator__learning_rate': [0.05],
    'classifier__estimator__subsample': [0.8],
    'classifier__estimator__colsample_bytree': [0.8],
    'classifier__estimator__min_child_weight': [5],
    'classifier__estimator__gamma': [0],
    'classifier__estimator__reg_alpha': [0],
    'classifier__estimator__reg_lambda': [1],
}

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z']
)

n_classes = sliced_data_df['gesture'].nunique()

xgb_clf = XGBClassifier(
    objective='multi:softmax',
    num_class=n_classes,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator = xgb_clf, 
        extractor = custom_extractor, 
        mode = 'xgboost'
        ))
])

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

y = sliced_data_df[['sequence_id', 'gesture']]
n_fits = len(list(ParameterGrid(param_grid))) * n_splits

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])
pbar.close()

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)

In [ ]:
cv_results_df.T

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
dc_offset_max = 10

pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']

model_run_name = 'xgboost.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

train_sample_pct = 0.20
test_sample_pct = 0.05

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

In [ ]:
train_df = raw_train_df.set_index('row_id')